# Makeitalive - Google Colab Runner

This notebook allows you to easily clone the GitHub repository, install dependencies, extract a dataset, train the motion flow model, and download the resulting weights.

**Note:** To speed up training, make sure you are using a GPU runtime (`Runtime` -> `Change runtime type` -> `T4 GPU`).

## 1. Setup Environment

In [ ]:
!git clone https://github.com/matu1003/Makeitalive.git
%cd Makeitalive
!pip install -e .

import sys
import os
# Ensure the src directory is in the Python path for easy imports
sys.path.append(os.path.abspath('src'))

## 2. Dataset Collection

In [ ]:
from data.download_youtube import download_video
from data.make_dataset_video import extract_pairs_from_video

video_path = "./data/10hourslandscape.mp4"
dataset_dir = os.path.join(".", "data", "dataset_local")

print("1. Downloading video...")
download_video(
    url="https://www.youtube.com/watch?v=AKeUssuu3Is",
    output_path=video_path,
    max_height=720
)

print("\n2. Extracting dataset pairs...")
extract_pairs_from_video(
    video_path=video_path,
    output_dir=dataset_dir,
    sample_every_n_seconds=5.0,
    frame_gap=4,
    target_size=512,
    max_pairs=100,
    clean=True
)

## 3. Training the Model
Run the training loop on the dataset generated above.

In [ ]:
import argparse
from motion_flow.train import train

# Create an arguments namespace equivalent to Python argparse
args = argparse.Namespace(
    data_dir=dataset_dir,
    ckpt_dir="./checkpoints",
    epochs=10,
    batch_size=8,
    lr=1e-4,
    num_workers=2
)

print("Starting training...")
train(args)

## 4. Evaluation and Export
Once the training is completed, run this cell to pack the checkpoints into a ZIP archive and download it to your local machine.

In [ ]:
import shutil
from google.colab import files

# Compress the checkpoints directory
shutil.make_archive('checkpoints_archive', 'zip', 'checkpoints')

# Trigger a download to save the weights locally
files.download('checkpoints_archive.zip')